# 0.3 TensorFlow 專案標準流程

這份 Notebook 使用 Breast Cancer 二元分類資料集，示範一個 TensorFlow/Keras 專案從資料到模型儲存的完整流程。

## 1. 載入套件

本範例會用 sklearn 載入資料與計算評估指標，用 TensorFlow/Keras 建立神經網路。

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

tf.keras.utils.set_random_seed(42)

## 2. 載入資料

Breast Cancer 是 sklearn 內建的二元分類資料集。每筆資料包含多個數值特徵，目標是判斷腫瘤類別。這裡用它示範標準 supervised learning 流程。

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df['target_name'] = df['target'].map(dict(enumerate(data.target_names)))

print('資料筆數與欄位數:', df.shape)
print('target names:', data.target_names)
df.head()

## 3. 切分資料

先切出測試集，再從訓練資料中切出驗證集。測試集保留到最後評估，不參與訓練與調參。

In [ ]:
X = data.data.values.astype('float32')
y = data.target.values.astype('float32')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

print('x_train:', x_train.shape)
print('x_valid:', x_valid.shape)
print('x_test:', x_test.shape)

## 4. 前處理資料

標準化只使用訓練集 `fit`，再套用到驗證集與測試集，避免測試資料資訊提前洩漏。

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

pd.DataFrame(x_train, columns=data.feature_names).describe().loc[['mean', 'std']].round(3)

## 5. 建立模型

這是一個基礎 Dense DNN。二元分類的輸出層使用 1 個神經元搭配 `sigmoid`，輸出可視為屬於正類的機率。

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

## 6. 編譯模型

二元分類常用 `binary_crossentropy`，評估指標除了 accuracy，也加入 AUC 觀察分類排序能力。

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

## 7. 訓練模型

加入 EarlyStopping，當驗證集 loss 不再改善時提早停止，並還原到最佳權重。

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True
    )
]

history = model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=0
)

print('實際訓練 epochs:', len(history.history['loss']))

## 8. 評估模型

先用 `evaluate()` 取得整體分數，再用 `predict()` 產生每筆資料的預測機率與類別。

In [ ]:
train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
valid_result = model.evaluate(x_valid, y_valid, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, valid_result, test_result], index=['train', 'valid', 'test'])

In [ ]:
prob = model.predict(x_test, verbose=0).ravel()
y_pred = (prob >= 0.5).astype(int)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=data.target_names, digits=4))
print('ROC AUC:', round(roc_auc_score(y_test, prob), 4))

## 9. 觀察訓練曲線

Learning curve 可以幫助判斷模型是否持續改善、是否過擬合，或是否需要調整 learning rate 與模型容量。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='valid loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curve')
plt.legend()
plt.show()

## 10. 儲存與載入模型

訓練完成後，將模型存成 `.keras` 檔。載入後可以直接再次推論，不需要重新訓練。

In [ ]:
OUTPUT_DIR = Path('outputs')
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / 'breast_cancer_dnn.keras'
model.save(model_path)

loaded_model = tf.keras.models.load_model(model_path)
loaded_prob = loaded_model.predict(x_test[:5], verbose=0).ravel()

pd.DataFrame({
    'actual': y_test[:5].astype(int),
    'probability': loaded_prob,
    'predicted': (loaded_prob >= 0.5).astype(int)
})

## 11. 小結

這份 Notebook 展示了 TensorFlow 專案的標準流程：載入資料、切分資料、前處理、建立模型、編譯、訓練、評估、預測、儲存與載入。後續 cookbook 任務會依照相同邏輯延伸。